In [1]:
import pandas as pd
import os
from abc import ABC, abstractmethod
from typing import Union, List
from pydantic.dataclasses import dataclass
import sys
from pathlib import Path

# Adiciona o diretório pai (WeatherOps) ao sys.path
caminho_raiz = str(Path.cwd().parent)
if caminho_raiz not in sys.path:
    sys.path.append(caminho_raiz)



In [2]:
from core.data_engineering import DataCleaning

In [3]:
os.listdir('../data/raw/2024/')[:5]

['INMET_CO_DF_A001_BRASILIA_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A042_BRAZLANDIA_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2024_A_31-12-2024.CSV']

In [3]:
csv_path = '../data/raw/2024/INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV'
df = pd.read_csv(csv_path, sep=';', encoding='latin-1', skiprows=lambda x: x in range(8))
 
df.head()

,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2024/01/01,0000 UTC,0,"1006,7","1006,7","1005,8",NaN,"26,8","22,8","26,8","26,6","22,8","22,6",79.0,78.0,79.0,66.0,"5,3","1,4",NaN
1,2024/01/01,0100 UTC,0,"1006,9","1006,9","1006,7",NaN,"26,7","22,6","26,8","26,5","22,8","22,4",79.0,77.0,78.0,62.0,"5,5","1,2",NaN
2,2024/01/01,0200 UTC,0,"1006,9","1007,1","1006,8",NaN,"26,5","22,9","26,7","26,4",23,"22,5",81.0,78.0,81.0,75.0,"4,6","1,2",NaN
3,2024/01/01,0300 UTC,0,"1006,5","1006,9","1006,5",NaN,"26,3","22,7","26,5","26,2","22,9","22,6",82.0,80.0,81.0,69.0,"4,6","1,1",NaN
4,2024/01/01,0400 UTC,0,"1006,5","1006,6","1006,5",NaN,26,"22,6","26,3","25,7","22,7","22,5",82.0,81.0,81.0,56.0,"4,2",",8",NaN


In [4]:
class IDataCleaning(ABC):

    @abstractmethod
    def _load_all_data(self):
        pass

    @abstractmethod
    def _default_read_csv(self):
        pass

    @abstractmethod
    def _cleaning_str_data_hours_columns(self):
        pass
    

In [5]:
@dataclass
class DataEngInput:
    csv_paths: List[str]

class DataCleaning(IDataCleaning):
    def __init__(self, csv_paths:Union[str, List[str]]):

        paths = [csv_paths] if isinstance(csv_paths, str) else csv_paths
        self.input = DataEngInput(csv_paths=paths)

        self.raw_dataframes = self._load_all_data()
        self.process_data()

    def _default_read_csv(self,path:str) -> pd.DataFrame:
        _df = pd.read_csv(path, sep=';', encoding='latin-1', skiprows=lambda x: x in range(8))
        return _df.iloc[:,:-1]

    def _load_all_data(self) -> List[pd.DataFrame]:
        return [self._default_read_csv(path) for path in self.input.csv_paths]

    def _cleaning_str_data_hours_columns(self, df:pd.DataFrame) -> pd.DataFrame:
        df['DATA_HORA'] = df['Data'] + df['Hora UTC'].str.strip('UTC').str.strip(' ')
        df['DATA_HORA'] = pd.to_datetime(df['DATA_HORA'], format='%Y/%m/%d%H%M')
        return df.drop(['Data', 'Hora UTC'], axis=1)
    

    def _convert_str_to_numeric(self, df:pd.DataFrame, dtype:List[str]=['object', 'string']) -> pd.DataFrame:
        col_strings_type = df.select_dtypes(include=dtype).columns
        for col in col_strings_type:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.', regex=False), errors='coerce')
        return df
    
    def _rename_all_columns(self, df:pd.DataFrame) -> pd.DataFrame:
        rename_map = {
            'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao_total_mm',
            'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)': 'pressao_atm_estacao_mb',
            'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)': 'pressao_atm_max_mb',
            'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)': 'pressao_atm_min_mb',
            'RADIACAO GLOBAL (Kj/m²)': 'radiacao_global_kj_m2',
            'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temp_ar_c',
            'TEMPERATURA DO PONTO DE ORVALHO (°C)': 'temp_ponto_orvalho_c',
            'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temp_max_c',
            'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temp_min_c',
            'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)': 'temp_orvalho_max_c',
            'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)': 'temp_orvalho_min_c',
            'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)': 'umidade_rel_max_percent',
            'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)': 'umidade_rel_min_percent',
            'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_rel_ar_percent',
            'VENTO, DIREÇÃO HORARIA (gr) (° (gr))': 'vento_direcao_graus',
            'VENTO, RAJADA MAXIMA (m/s)': 'vento_rajada_ms',
            'VENTO, VELOCIDADE HORARIA (m/s)': 'vento_vel_ms',
            'DATA_HORA': 'data_hora'
        }

        return df.rename(columns=rename_map)

    def process_data(self) -> List[pd.DataFrame]:
        """Aplica a transformação em toda a lista de dataframes."""
        self.raw_dataframes = [self._cleaning_str_data_hours_columns(df) for df in self.raw_dataframes]
        self.raw_dataframes = [self._convert_str_to_numeric(df) for df in self.raw_dataframes]
        self.raw_dataframes = [self._rename_all_columns(df) for df in self.raw_dataframes]

    def concat_csv(self) -> pd.DataFrame:
        return pd.concat(self.raw_dataframes)
            


In [6]:
from pathlib import Path
from typing import List, Dict

def map_csv_files_by_name(root_path: str, search_names: List[str]) -> Dict[str, List[str]]:
    """
    Percorre pastas recursivamente buscando arquivos .csv que contenham 
    os nomes da lista no título.
    """
    root = Path(root_path)
    
    
    result = {name: [] for name in search_names}
    
    
    for csv_file in root.rglob('*.csv'):
        
        file_name_lower = csv_file.name.lower()
        
        for search_name in search_names:
            
            if search_name.lower() in file_name_lower:
                result[search_name].append(str(csv_file.absolute()))
                
    return result

test_dict = map_csv_files_by_name(root_path='../data/raw/', search_names=['salvador_'])

In [7]:
test_dict

{'salvador_': ['c:\\Users\\MaquinaLegal\\projetos\\WeatherOps\\notebooks\\..\\data\\raw\\2024\\INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV',
  'c:\\Users\\MaquinaLegal\\projetos\\WeatherOps\\notebooks\\..\\data\\raw\\2025\\INMET_NE_BA_A401_SALVADOR_01-01-2025_A_31-12-2025.CSV',
  'c:\\Users\\MaquinaLegal\\projetos\\WeatherOps\\notebooks\\..\\data\\raw\\2026\\INMET_NE_BA_A401_SALVADOR_01-01-2026_A_28-02-2026.CSV']}

In [8]:
data_test = DataCleaning(csv_paths=test_dict['salvador_'])


In [9]:
df_final = data_test.concat_csv()

In [10]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 18960 entries, 0 to 1415
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   precipitacao_total_mm    18934 non-null  float64       
 1   pressao_atm_estacao_mb   18934 non-null  float64       
 2   pressao_atm_max_mb       18934 non-null  float64       
 3   pressao_atm_min_mb       18934 non-null  float64       
 4   radiacao_global_kj_m2    10268 non-null  float64       
 5   temp_ar_c                18934 non-null  float64       
 6   temp_ponto_orvalho_c     18934 non-null  float64       
 7   temp_max_c               18934 non-null  float64       
 8   temp_min_c               18934 non-null  float64       
 9   temp_orvalho_max_c       18934 non-null  float64       
 10  temp_orvalho_min_c       18934 non-null  float64       
 11  umidade_rel_max_percent  18934 non-null  float64       
 12  umidade_rel_min_percent  18934 non-null  float64 

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

class AnaliseClimaticaEDA:
    def __init__(self, df): 
        """
        Inicializa a classe, cria uma cópia do DataFrame para não alterar o original,
        garante que a coluna de data seja datetime e ordena os dados.
        """
        self.df = df.copy()
        
        
        if 'data_hora' in self.df.columns:
            self.df['data_hora'] = pd.to_datetime(self.df['data_hora'])
            self.df = self.df.sort_values(by='data_hora')
        else:
            print("Atenção: Coluna 'data_hora' não encontrada. Alguns gráficos de linha podem falhar.")

    def plot_serie_temporal(self, coluna, titulo=None):
        """Gera um gráfico de linha interativo para uma variável ao longo do tempo."""
        if titulo is None:
            titulo = f"Série Temporal: {coluna}"
            
        fig = px.line(self.df, x='data_hora', y=coluna, title=titulo,
                      labels={'data_hora': 'Data e Hora', coluna: coluna})
        
        
        fig.update_xaxes(rangeslider_visible=True)
        fig.show()

    def plot_temperaturas_conjuntas(self):
        """Plota as temperaturas Máxima, Mínima e Média/Ar no mesmo gráfico."""
        fig = go.Figure()

        fig.add_trace(go.Scatter(x=self.df['data_hora'], y=self.df['temp_max_c'],
                                 mode='lines', name='Temp Máx', line=dict(color='red')))
        fig.add_trace(go.Scatter(x=self.df['data_hora'], y=self.df['temp_ar_c'],
                                 mode='lines', name='Temp Ar', line=dict(color='green')))
        fig.add_trace(go.Scatter(x=self.df['data_hora'], y=self.df['temp_min_c'],
                                 mode='lines', name='Temp Mín', line=dict(color='blue')))

        fig.update_layout(title='Dinâmica Diária das Temperaturas (°C)',
                          xaxis_title='Data e Hora',
                          yaxis_title='Temperatura (°C)',
                          hovermode="x unified") 
        fig.show()

    def plot_distribuicao(self, coluna):
        """Gera um histograma com boxplot marginal para ver a distribuição dos dados."""
        fig = px.histogram(self.df, x=coluna, marginal="box", 
                           title=f"Distribuição de {coluna}",
                           labels={coluna: coluna})
        fig.show()

    def plot_matriz_correlacao(self):
        """Gera um heatmap de correlação (Pearson) entre as variáveis numéricas."""
        # Seleciona apenas as colunas numéricas, removendo a data
        df_numerico = self.df.select_dtypes(include=['float64', 'int64'])
        correlacao = df_numerico.corr()

        fig = px.imshow(correlacao, text_auto='.2f', aspect="auto",
                        color_continuous_scale='RdBu_r', 
                        title="Matriz de Correlação das Variáveis Meteorológicas")
        fig.show()
        
    def resumo_dados_ausentes(self):
        """Retorna um DataFrame mostrando o percentual de dados nulos por coluna."""
        nulos = self.df.isnull().sum()
        percentual = (nulos / len(self.df)) * 100
        resumo = pd.DataFrame({'Nulos': nulos, 'Percentual (%)': percentual}).sort_values(by='Nulos', ascending=False)
        return resumo[resumo['Nulos'] > 0]

In [12]:
eda = AnaliseClimaticaEDA(df_final)

print(eda.resumo_dados_ausentes())

eda.plot_serie_temporal('temp_ar_c')

eda.plot_temperaturas_conjuntas()

eda.plot_distribuicao('temp_ar_c')

eda.plot_matriz_correlacao()

                         Nulos  Percentual (%)
radiacao_global_kj_m2     8692       45.843882
precipitacao_total_mm       26        0.137131
pressao_atm_estacao_mb      26        0.137131
pressao_atm_max_mb          26        0.137131
pressao_atm_min_mb          26        0.137131
temp_ar_c                   26        0.137131
temp_ponto_orvalho_c        26        0.137131
temp_max_c                  26        0.137131
temp_min_c                  26        0.137131
temp_orvalho_max_c          26        0.137131
temp_orvalho_min_c          26        0.137131
umidade_rel_max_percent     26        0.137131
umidade_rel_min_percent     26        0.137131
umidade_rel_ar_percent      26        0.137131
vento_direcao_graus         26        0.137131
vento_rajada_ms             26        0.137131
vento_vel_ms                26        0.137131


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

class AnaliseAnualEDA: 
    def __init__(self, df):
        """
        Inicializa a classe e prepara os dados extraindo Ano, Mês e Dia.
        """
        self.df = df.copy()
        
        if 'data_hora' in self.df.columns:
            self.df['data_hora'] = pd.to_datetime(self.df['data_hora'])
            self.df['ano'] = self.df['data_hora'].dt.year
            self.df['mes'] = self.df['data_hora'].dt.month
            self.df['nome_mes'] = self.df['data_hora'].dt.strftime('%b')
            self.df['dia_do_ano'] = self.df['data_hora'].dt.dayofyear
            self.df['hora'] = self.df['data_hora'].dt.hour
            self.df['data_sem_hora'] = self.df['data_hora'].dt.date
        else:
            raise ValueError("A coluna 'data_hora' é obrigatória para esta análise.")

    def comparar_distribuicao_anual(self, coluna):
        fig = px.box(self.df, x='ano', y=coluna, color='ano',
                     title=f'Comparação Anual - Distribuição de {coluna}',
                     labels={'ano': 'Ano', coluna: coluna})
        fig.update_xaxes(type='category')
        fig.show()

    def evolucao_mensal_por_ano(self, coluna, agregacao='mean'):
        if agregacao == 'mean':
            df_agregado = self.df.groupby(['ano', 'mes'])[coluna].mean().reset_index()
            titulo = f'Média Mensal de {coluna} por Ano'
        elif agregacao == 'sum':
            df_agregado = self.df.groupby(['ano', 'mes'])[coluna].sum().reset_index()
            titulo = f'Total Acumulado Mensal de {coluna} por Ano'
        else:
            raise ValueError("Agregação deve ser 'mean' ou 'sum'.")

        fig = px.line(df_agregado, x='mes', y=coluna, color='ano', markers=True,
                      title=titulo, labels={'mes': 'Mês', coluna: coluna})
        fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)
        fig.update_traces(line=dict(width=3))
        fig.show()

    def acumulado_anual_precipitacao(self):
        if 'precipitacao_total_mm' not in self.df.columns:
            print("A coluna 'precipitacao_total_mm' não existe no DataFrame.")
            return

        df_chuva = self.df.groupby('ano')['precipitacao_total_mm'].sum().reset_index()
        fig = px.bar(df_chuva, x='ano', y='precipitacao_total_mm', text_auto='.1f',
                     title='Precipitação Total Acumulada por Ano',
                     labels={'ano': 'Ano', 'precipitacao_total_mm': 'Chuva Total (mm)'},
                     color='precipitacao_total_mm', color_continuous_scale='Blues')
        fig.update_xaxes(type='category')
        fig.show()

    def perfil_horario_por_ano(self, coluna, agregacao='mean'):
        """
        Gráfico de linhas mostrando o perfil médio (ou somado) hora a hora,
        com uma linha por ano. Ideal para comparar padrões diários entre anos
        — ex.: temperatura média às 14h foi maior em 2023 do que em 2022?

        Parâmetros
        ----------
        coluna     : variável a analisar
        agregacao  : 'mean' para médias (temperatura, umidade…)
                     'sum'  para totais (precipitação, radiação…)
        """
        func = 'mean' if agregacao == 'mean' else 'sum'
        df_hora = (
            self.df.groupby(['ano', 'hora'])[coluna]
            .agg(func)
            .reset_index()
        )
        df_hora['ano'] = df_hora['ano'].astype(str)   # cor discreta na legenda

        titulo = (
            f'Perfil Horário — {"Média" if func == "mean" else "Total"} de {coluna} por Ano'
        )
        fig = px.line(
            df_hora, x='hora', y=coluna, color='ano',
            markers=True, title=titulo,
            labels={'hora': 'Hora do dia (0–23)', coluna: coluna, 'ano': 'Ano'},
        )
        fig.update_xaxes(tickmode='linear', tick0=0, dtick=1, range=[-0.5, 23.5])
        fig.update_traces(line=dict(width=2.5))
        fig.update_layout(hovermode='x unified')
        fig.show()

    def distribuicao_horaria_por_ano(self, coluna):
        """
        Boxplot por hora do dia, com facetas (subplots) separadas por ano.
        Permite ver, em cada ano, como a variabilidade dentro do dia se distribui
        — ex.: a dispersão de temperatura às 15h é maior no verão de 2024?

        A figura fica mais alta conforme o número de anos aumenta.
        """
        anos = sorted(self.df['ano'].unique())
        n_anos = len(anos)

        fig = px.box(
            self.df, x='hora', y=coluna,
            facet_row='ano',
            color='ano',
            title=f'Distribuição por Hora do Dia — {coluna} (por Ano)',
            labels={'hora': 'Hora', coluna: coluna, 'ano': 'Ano'},
            height=300 * n_anos,
        )
        fig.update_xaxes(tickmode='linear', tick0=0, dtick=1)
        fig.update_layout(showlegend=False)
        fig.show()

    def evolucao_diaria_por_ano(self, coluna, agregacao='mean'):
        """
        Gráfico de linhas com a evolução dia a dia ao longo do ano (eixo X =
        dia do ano, 1–366), com uma linha por ano.
        Permite alinhar os anos num mesmo eixo temporal e comparar diretamente
        — ex.: o dia 200 (19/jul) foi mais quente em 2022 ou 2023?

        Parâmetros
        ----------
        coluna     : variável a analisar
        agregacao  : 'mean' | 'sum'
        """
        func = 'mean' if agregacao == 'mean' else 'sum'
        df_dia = (
            self.df.groupby(['ano', 'dia_do_ano'])[coluna]
            .agg(func)
            .reset_index()
        )
        df_dia['ano'] = df_dia['ano'].astype(str)

        titulo = (
            f'Evolução Diária — {"Média" if func == "mean" else "Total"} '
            f'de {coluna} por Ano'
        )
        fig = px.line(
            df_dia, x='dia_do_ano', y=coluna, color='ano',
            title=titulo,
            labels={'dia_do_ano': 'Dia do Ano (1–366)', coluna: coluna, 'ano': 'Ano'},
        )
        fig.update_traces(line=dict(width=1.5), opacity=0.85)
        fig.update_layout(hovermode='x unified')
        fig.show()

    def heatmap_hora_mes_por_ano(self, coluna, agregacao='mean'):
        """
        Heatmap 12×24 (mês × hora) com uma aba (subplot) por ano.
        Combina as duas granularidades (hora e mês) numa única visualização,
        deixando evidente sazonalidade intra-dia e intra-anual simultaneamente.
        Excelente para variáveis como temperatura, umidade e radiação solar.

        Parâmetros
        ----------
        coluna     : variável a analisar
        agregacao  : 'mean' | 'sum'
        """
        from plotly.subplots import make_subplots

        func = 'mean' if agregacao == 'mean' else 'sum'
        label_z = f'{"Média" if func == "mean" else "Total"} de {coluna}'

        df_hm = (
            self.df.groupby(['ano', 'mes', 'hora'])[coluna]
            .agg(func)
            .reset_index()
        )

        anos = sorted(df_hm['ano'].unique())
        n_anos = len(anos)

        # Calcula a escala de cor global para que todos os anos sejam comparáveis
        z_min = df_hm[coluna].min()
        z_max = df_hm[coluna].max()

        fig = make_subplots(
            rows=1, cols=n_anos,
            subplot_titles=[str(a) for a in anos],
            shared_yaxes=True,
        )

        meses_abrev = ['Jan','Fev','Mar','Abr','Mai','Jun',
                       'Jul','Ago','Set','Out','Nov','Dez']

        for i, ano in enumerate(anos, start=1):
            pivot = (
                df_hm[df_hm['ano'] == ano]
                .pivot(index='mes', columns='hora', values=coluna)
                .reindex(index=range(1, 13), columns=range(0, 24))
            )

            heatmap = go.Heatmap(
                z=pivot.values,
                x=[f'{h:02d}h' for h in range(24)],
                y=meses_abrev,
                colorscale='RdBu_r',
                zmin=z_min, zmax=z_max,
                colorbar=dict(title=label_z) if i == n_anos else dict(showticklabels=False),
                showscale=(i == n_anos),      # mostra barra de cor só no último subplot
                hovertemplate='Mês: %{y}<br>Hora: %{x}<br>Valor: %{z:.2f}<extra></extra>',
            )
            fig.add_trace(heatmap, row=1, col=i)

        fig.update_layout(
            title_text=f'Heatmap Hora × Mês — {label_z}',
            height=420,
            width=350 * n_anos,
        )
        fig.show()

    def acumulado_anual_precipitacao(self):
        """
        Gera um gráfico de barras com o volume total de chuva (precipitação) por ano.
        """
        if 'precipitacao_total_mm' not in self.df.columns:
            print("A coluna 'precipitacao_total_mm' não existe no DataFrame.")
            return

        df_chuva = self.df.groupby('ano')['precipitacao_total_mm'].sum().reset_index()

        fig = px.bar(df_chuva, x='ano', y='precipitacao_total_mm', text_auto='.1f',
                     title='Precipitação Total Acumulada por Ano',
                     labels={'ano': 'Ano', 'precipitacao_total_mm': 'Chuva Total (mm)'},
                     color='precipitacao_total_mm', color_continuous_scale='Blues')
        
        fig.update_xaxes(type='category')
        fig.show()

    def contar_dias_extremos(self, temperatura_limite, tipo='max'):
        """
        Conta quantos dias por ano a temperatura ultrapassou um limite (calor) 
        ou ficou abaixo de um limite (frio).
        
        tipo: 'max' (para dias mais quentes que o limite) ou 'min' (para dias mais frios)
        """
        # CORREÇÃO AQUI: Note os colchetes duplos [['temp_max_c', 'temp_min_c']]
        df_diario = self.df.groupby(['ano', 'data_sem_hora'])[['temp_max_c', 'temp_min_c']].agg(
            temp_max_c=('temp_max_c', 'max'),
            temp_min_c=('temp_min_c', 'min')
        ).reset_index()

        if tipo == 'max':
            dias_extremos = df_diario[df_diario['temp_max_c'] >= temperatura_limite]
            titulo = f'Número de Dias no Ano com Temperatura ≥ {temperatura_limite}°C'
        else:
            dias_extremos = df_diario[df_diario['temp_min_c'] <= temperatura_limite]
            titulo = f'Número de Dias no Ano com Temperatura ≤ {temperatura_limite}°C'

        contagem = dias_extremos.groupby('ano').size().reset_index(name='qtd_dias')
        
        # Garante que anos com 0 dias extremos apareçam no gráfico
        todos_os_anos = pd.DataFrame({'ano': self.df['ano'].unique()})
        contagem = todos_os_anos.merge(contagem, on='ano', how='left').fillna(0)

        fig = px.bar(contagem, x='ano', y='qtd_dias', text_auto=True,
                     title=titulo, labels={'ano': 'Ano', 'qtd_dias': 'Quantidade de Dias'})
        
        fig.update_xaxes(type='category')
        fig.update_traces(marker_color='red' if tipo == 'max' else 'blue')
        fig.show()

    

In [14]:
analise_anos = AnaliseAnualEDA(df_final)

analise_anos.comparar_distribuicao_anual('temp_ar_c')

analise_anos.evolucao_mensal_por_ano('temp_ar_c', agregacao='mean')

analise_anos.evolucao_mensal_por_ano('temp_ar_c', agregacao='sum')

analise_anos.acumulado_anual_precipitacao()

analise_anos.contar_dias_extremos(temperatura_limite=32, tipo='max')

analise_anos.perfil_horario_por_ano('temp_ar_c')

analise_anos.distribuicao_horaria_por_ano('temp_ar_c')

analise_anos.evolucao_diaria_por_ano('temp_ar_c')

analise_anos.heatmap_hora_mes_por_ano('temp_ar_c')

In [ ]:



teste = DataCleaning('c:\\Users\\MaquinaLegal\\projetos\\WeatherOps\\notebooks\\..\\data\\raw\\2024\\INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV')

In [22]:
teste.raw_dataframes

[      precipitacao_total_mm  pressao_atm_estacao_mb  pressao_atm_max_mb  \
 0                       0.0                  1006.7              1006.7   
 1                       0.0                  1006.9              1006.9   
 2                       0.0                  1006.9              1007.1   
 3                       0.0                  1006.5              1006.9   
 4                       0.0                  1006.5              1006.6   
 ...                     ...                     ...                 ...   
 8779                    0.0                  1005.9              1006.0   
 8780                    0.0                  1005.8              1005.9   
 8781                    0.0                  1005.9              1006.0   
 8782                    0.0                  1006.5              1006.5   
 8783                    0.0                  1006.9              1006.9   
 
       pressao_atm_min_mb  radiacao_global_kj_m2  temp_ar_c  \
 0                 1005